In [1]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI()


# 1. Request validation
class ChatRequest(BaseModel):
    question: str = Field(
        min_length=3,
        max_length=1000
    )


# 2. Structured response
class ChatResponse(BaseModel):
    answer: str
    status: str


# Placeholder for the actual LLM call
def call_llm(question: str) -> str:
    # In production, call GPT or Llama here
    return f"AI generated answer for: {question}"


@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):

    try:
        # Call the LLM
        answer = call_llm(request.question)

        # Return structured JSON
        return ChatResponse(
            answer=answer,
            status="success"
        )

    except Exception:
        raise HTTPException(
            status_code=500,
            detail="Failed to generate AI response"
        )

In [5]:
import os

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings
)
from llama_index.embeddings.openai import OpenAIEmbedding

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


# 1. Configure embedding model
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)


# 2. Load enterprise documents
documents = SimpleDirectoryReader(
    input_dir="./data"
).load_data()


# 3. Create vector index
index = VectorStoreIndex.from_documents(
    documents
)


# 4. Create retriever using LlamaIndex
retriever = index.as_retriever(
    similarity_top_k=3
)


# 5. Configure LangChain LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# 6. Create prompt template
prompt = ChatPromptTemplate.from_template("""
You are an enterprise AI assistant.

Answer the question using only the
provided context.

If the answer is not in the context,
say you don't know.

Context:
{context}

Question:
{question}
""")


# 7. RAG connection function
def ask_question(question):

    # Retrieve documents using LlamaIndex
    nodes = retriever.retrieve(question)

    # Extract retrieved text
    context = "\n\n".join(
        node.get_content()
        for node in nodes
    )

    # Pass context to LangChain prompt
    messages = prompt.format_messages(
        context=context,
        question=question
    )

    # Generate answer using LangChain
    response = llm.invoke(messages)

    return response.content


# 8. Ask a question
answer = ask_question(
    "How many annual leave days are available?"
)

print(answer)

ModuleNotFoundError: No module named 'llama_index'

In [6]:
import sys

print(sys.executable)

import llama_index
print("LlamaIndex installed successfully!")

d:\LLM_Projects\llm_engineering\.venv\Scripts\python.exe


ModuleNotFoundError: No module named 'llama_index'